# Final Notebook

## Analyst Decisions

- Creating a Pivot Chart for each Brench containing the number of events for each catagory:
    1. Children: 0 to 12
    2. Teens : 13 to 17
    3. Adults : 18+

***Note: Might need to reclassify the Adults into three categories: Young Adults, Adults, and Seniors.***

- Then, making a conclusion page that will include:
   - Top 15 Branches
   - Lowest 15 branches
   - Librarys that have more space/ computer hubs (Resources)
   - Regestiration to the library: how many children/Adults are registered



In [4]:
import sys
print(sys.executable)

/home/eden/U+/UVolunteer-/venv/bin/python


In [5]:
# Import Lib

import pandas as pd
import sweetviz as sv
import matplotlib.pyplot as plt #this is for Graph creation.


# Explore DF function
def explore (Data):
    print(Data.head(5))
    print(Data.info())
    M_Values = Data.isnull().sum().sort_values(ascending=False) 
    print(M_Values)
    print(Data.duplicated().sum())
    missing = M_Values.index[0]
    print(Data[Data[missing].isnull()])

# DataFrame creation

tpl_BGI_2023 = pd.read_csv("data/tpl-branch-general-information-2023.csv",index_col=0)
tpl_BSR_2024 = pd.read_csv("data/tpl-branch-space-rentals-2024.csv",index_col=0)
tpl_RABB = pd.read_csv("data/tpl-card-registrations-annual-by-branch.csv",index_col=0)
tpl_EFS2 = pd.read_csv("data/tpl-events-feed_source2.csv",index_col=0)
tpl_VABB = pd.read_csv("data/tpl-visits-annual-by-branch.csv",index_col=0)
tpl_WUABB_2018_2023 = pd.read_csv("data/tpl-workstation-usage-annual-by-branch-2018-2023.csv",index_col=0)
tpl_LCBCT = pd.read_csv("data/library-circulation-by-cardholder-type.csv",index_col=0)

tpl_BGI_2023.head(5)

,BranchCode,PhysicalBranch,BranchName,Address,PostalCode,Website,Telephone,SquareFootage,PublicParking,KidsStop,...,Workstations,ServiceTier,Lat,Long,NBHDNo,NBHDName,TPLNIA,WardNo,WardName,PresentSiteYear
_id,,,,,,,,,,,,,,,,,,,,,
1,AB,1,Albion,"1515 Albion Road, Toronto, ON, M9V 1B2",M9V 1B2,https://www.tpl.ca/albion,416-394-5170,29000,59,1.0,...,38.0,DL,43.739826,-79.584096,2.0,Mount Olive-Silverstone-Jamestown,1.0,1.0,Etobicoke North,2017.0
2,ACD,1,Albert Campbell,"496 Birchmount Road, Toronto, ON, M1K 1N8",M1K 1N8,https://www.tpl.ca/albertcampbell,416-396-8890,28957,45,0.0,...,36.0,DL,43.708019,-79.269252,120.0,Clairlea-Birchmount,1.0,20.0,Scarborough Southwest,1971.0
3,AD,1,Alderwood,"2 Orianna Drive, Toronto, ON, M8W 4Y1",M8W 4Y1,https://www.tpl.ca/alderwood,416-394-5310,7341,shared,0.0,...,7.0,NL,43.601944,-79.547252,20.0,Alderwood,0.0,3.0,Etobicoke-Lakeshore,1999.0
4,AG,1,Agincourt,"155 Bonis Avenue, Toronto, ON, M1T 3W6",M1T 3W6,https://www.tpl.ca/agincourt,416-396-8943,27000,86,0.0,...,42.0,DL,43.785167,-79.293430,118.0,Tam O'Shanter-Sullivan,0.0,22.0,Scarborough-Agincourt,1991.0
5,AH,1,Armour Heights,"2140 Avenue Road, Toronto, ON, M5M 4M7",M5M 4M7,https://www.tpl.ca/armourheights,416-395-5430,2988,shared,0.0,...,5.0,NL,43.739337,-79.421889,39.0,Bedford Park-Nortown,0.0,8.0,Eglinton-Lawrence,1982.0


## Cleaning Decisions:

#### For table ***tpl-branch-general-information-2023*** 
1. Clumns removed: 
    - Location:  
    'Address',
    'PostalCode',
    'Lat',
    'Long',
    'WardName',
    'NBHDNo',
    'WardNo',
    - Program Indicators: 
    'AdultLiteracyProgram',
    'LeadingReading',
    - Operational: 
    'SquareFootage',
    'PublicParking',
    'Website',
    'Telephone',
    'ServiceTier',
    'PresentSiteYear'

2. Raws that contain 'KidsStop' were kept, (might need to change)
3. Brench name: 'Jane/Dundas', was changed to its current name: 'Daniel G. Hill'
4. The new DF called: tpl_BGI_2023_cl 



In [17]:
def clean_branches(df):
    df =  df[df['KidsStop'].notnull()]
    drop_columns = [
    # Location
    'Address',
    'PostalCode',
    'Lat',
    'Long',
    'WardName',
    'NBHDNo',
    'WardNo',
    # Program Indicators
    'AdultLiteracyProgram',
    'LeadingReading',
    # Operational
    'SquareFootage',
    'PublicParking',
    'Website',
    'Telephone',
    'ServiceTier',
    'PresentSiteYear']
    df = df.drop(columns = drop_columns)
    return df

tpl_BGI_2023_cl = clean_branches(tpl_BGI_2023)

tpl_BGI_2023_cl['BranchName'] = tpl_BGI_2023_cl['BranchName'].str.replace(
    'Jane/Dundas', 'Daniel G. Hill'
)


tpl_BGI_2023_cl['BranchName'] = tpl_BGI_2023_cl['BranchName'].str.replace(
    'Perth/Dupont', 'Junction Triangle'
)

# branch_names = set(tpl_BGI_2023_cl['BranchName'].unique())
branch_names = set(tpl_BGI_2023_cl['BranchName'].astype(str).str.strip().str.lower())
# len(tpl_BGI_2023_cl['BranchName'].unique())
print(branch_names)
print(len(branch_names))

{'long branch', 'forest hill', 'taylor memorial', 'cedarbrae', 'pleasant view', 'evelyn gregory', 'north york central library', 'riverdale', 'maria a. shchuka', 'rexdale', 'college/shaw', 'parkdale', 'main street', 'city hall', 'gerrard/ashdale', 'amesbury park', 'albion', 'guildwood', 'brentwood', 'new toronto', 'wychwood', 'daniel g. hill', 'swansea memorial', 'mcgregor park', 'deer park', 'woodview park', 'lillian h. smith', 'davenport', 'woodside square', 'richview', 's. walter stewart', 'humber bay', 'mount pleasant', 'dufferin/st. clair', 'thorncliffe', 'humberwood', 'highland creek', 'queen/saulter', 'armour heights', 'malvern', 'don mills', 'maryvale', 'flemingdon park', 'st. james town', 'dawes road', 'spadina road', 'eatonville', 'goldhawk park', 'humber summit', 'st. clair/silverthorn', 'bendale', 'centennial', 'burrows hall', 'morningside', 'steeles', 'runnymede', 'toronto reference library', 'jones', 'annette street', 'yorkville', 'elmbrook park', 'downsview', 'oakwood vil

#### Investigation Notes 
- Perth/Dupont, was closed in 2025 and moved to a new location called 'Junction Triangle'

- Now Junction Triangle is at 305 Campbell

- Jane/ Dundas Name was changed to Daniel G. Hill

- Libraries that don't show up on the events table:
    - todmorden room, humber bay, dawes road, pleasant view (Temp Closed),flemingdon park (Temp Closed), centennial (Closed for 3 years), 


#### For table ***tpl-events-feed_source2*** 

1. Columns removed:
        'EventID',
        'StartTime',
        'EndTime',
        'StartDateLocal',
        'RegistrationClosed',
        'FeaturedImageUrl'

2. Have 96 Locations, when Nan is Online events

In [18]:
# Table tpl-events-feed_source2 clean 

# print(tpl_EFS2['LastUpdatedOn'].unique())

def clean_branches(df):
    # remove all non physical branches
    # df = df[df[''].notnull()]
    drop_columns = [
        'EventID',
        'StartTime',
        'EndTime',
        'StartDateLocal',
        'RegistrationClosed',
        'FeaturedImageUrl'
    ]
    df = df.drop(columns = drop_columns)
    # df = df[~df['LocationName'].str.contains('Junction', case=True)]
    return df

tpl_EFS2_cl = clean_branches(tpl_EFS2)
# print(tpl_EFS2_cl.head(5))
# print(tpl_EFS2_cl.info())


tpl_EFS2_cl['LocationName'] = tpl_EFS2_cl['LocationName'].str.replace("Dufferin/St.Clair", "Dufferin/St. Clair")

tpl_EFS2_cl[tpl_EFS2_cl['LocationName'].isnull()]
events_locations = set(tpl_EFS2_cl['LocationName'].astype(str).str.strip().str.lower())
events_locations
branch_names = {x.lower() for x in branch_names}
diff_a = branch_names - events_locations
print(diff_a)

{'flemingdon park', 'dawes road', 'todmorden room', 'centennial', 'humber bay', 'pleasant view'}
